# Time-series forecasting — ARIMA (AutoRegressive Integrated Moving Average)

_Modern Deep Learning & AI_

**Predict the next value from recent values and recent surprises, then give a range, not just a point.**

A time series is numbers in order over time: daily sales, hourly temperature, a stock price.

     ARIMA forecasts the next value from the recent past. Its name is three ideas: AR (autoregression), I (integration / differencing), MA (moving average).

---

This notebook is a practice scaffold for the **Time-series forecasting — ARIMA (AutoRegressive Integrated Moving Average)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

class LSTMForecaster(nn.Module):
    def __init__(self, in_dim=1, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)         # predict next value
    def forward(self, x):                         # x: (B, T, in_dim)
        out, _ = self.lstm(x)                     # out: (B, T, hidden)
        return self.head(out[:, -1, :])           # use last time step -> (B, 1)

model = LSTMForecaster()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

# synthetic batch: 16 windows of length 20 (univariate series)
seq = torch.sin(torch.linspace(0, 8, 21)).repeat(16, 1)
x = seq[:, :20].unsqueeze(-1)                     # (16, 20, 1) input window
y = seq[:, 20:21]                                 # (16, 1) next value

pred = model(x)
loss = nn.functional.mse_loss(pred, y)
opt.zero_grad(); loss.backward(); opt.step()
print(float(loss))

## Visualize the data & results

_On the classic airline-passenger series, does the forecast track the real demand, and does the truth stay inside the prediction interval?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Real Box & Jenkins international airline passengers, 1949-1960 (144 months,
# thousands of passengers).
air = np.array([112,118,132,129,121,135,148,148,136,119,104,118,
115,126,141,135,125,149,170,170,158,133,114,140,
145,150,178,163,172,178,199,199,184,162,146,166,
171,180,193,181,183,218,230,242,209,191,172,194,
196,196,236,235,229,243,264,272,237,211,180,201,
204,188,235,227,234,264,302,293,259,229,203,229,
242,233,267,269,270,315,364,347,312,274,237,278,
284,277,317,313,318,374,413,405,355,306,271,306,
315,301,356,348,355,422,465,467,404,347,305,336,
340,318,362,348,363,435,491,505,404,359,310,337,
360,342,406,396,420,472,548,559,463,407,362,405,
417,391,419,461,472,535,622,606,508,461,390,432], dtype=float)

# Fit AR(2) on the log series by least squares (multiplicative trend + season).
ly = np.log(air)
T = len(air)
Xr = np.column_stack([np.ones(T-2), ly[1:-1], ly[:-2]])
coef, *_ = np.linalg.lstsq(Xr, ly[2:], rcond=None)
c, a1, a2 = coef                          # c=0.232, a1=1.174, a2=-0.215
sigma = (ly[2:] - Xr @ coef).std(ddof=3)  # residual std for the interval

idx = np.arange(T-24, T)                   # forecast the last 24 months
fc_l = c + a1 * ly[idx-1] + a2 * ly[idx-2]
fc = np.exp(fc_l)                           # back-transform to passenger counts
lower = np.exp(fc_l - 1.96 * sigma); upper = np.exp(fc_l + 1.96 * sigma)

plt.figure(figsize=(7, 4))
plt.plot(idx, air[idx], color='#4ea1ff', marker='o', label='actual')
plt.plot(idx, fc, color='#ffb454', marker='o', label='forecast')
plt.fill_between(idx, lower, upper, color='#c89bff', alpha=0.3, label='95% interval')
plt.xlabel('month index (1949-1960)'); plt.ylabel('thousands of passengers')
plt.title('Airline passengers: AR(2) forecast vs actual')
plt.legend(); plt.tight_layout(); plt.show()